#  Online Shoppers Intention Prediction using Machine Learning

##  Project Overview
This project predicts whether an online shopping session will result in revenue (purchase) or not using machine learning models.  
The dataset contains user behavior, session details, and browsing patterns from an e-commerce platform.

## Objective
To build a robust ML pipeline with preprocessing, feature engineering, and model selection using ROC-AUC as the primary evaluation metric.

## Step 1: Import Required Libraries
In this step, we import all the necessary Python libraries for:
- Data manipulation (Pandas, NumPy)
- Machine Learning (Scikit-learn)
- Preprocessing & Pipelines
- Model Evaluation Metrics

In [1]:
# Step 1: Import Required Libraries
import numpy as np
import time
import pandas as pd


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, chi2
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, f1_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB
from imblearn.pipeline import Pipeline as IMBPipeline  
from sklearn.pipeline import Pipeline  

import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")


Libraries Imported Successfully


## Step 2: Loading the Dataset
The dataset `online_shoppers_intention.csv` is loaded using Pandas.  
This dataset contains various features related to user sessions such as:
- Page values
- Bounce rates
- Visitor type
- Month
- Weekend activity
- Revenue (Target Variable)

In [2]:
#loading the dataset
df=pd.read_csv("..\\data\\online_shoppers_intention.csv")
print(df)

       Administrative  Administrative_Duration  Informational  \
0                   0                      0.0              0   
1                   0                      0.0              0   
2                   0                      0.0              0   
3                   0                      0.0              0   
4                   0                      0.0              0   
...               ...                      ...            ...   
12325               3                    145.0              0   
12326               0                      0.0              0   
12327               0                      0.0              0   
12328               4                     75.0              0   
12329               0                      0.0              0   

       Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                         0.0               1                 0.000000   
1                         0.0               2                64.000000 

## Step 3: Understanding the Dataset
In this step, we explore the dataset structure by:
- Viewing column names
- Checking data types
- Inspecting missing values
- Displaying the first few rows

This helps in understanding feature distribution and data quality before preprocessing.

In [3]:
#understanding the datset
print("columns in a dataset")
print(df.columns)
print("dataset information")
print(df.info())
print("printing 1st 5rows")
print(df.head())


columns in a dataset
Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')
dataset information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates 

##  Step 4: Checking Missing Values
We check for null or missing values in the dataset to ensure data quality.  
Handling missing values is important for building a stable machine learning model.

In [4]:
#checking the missing values 
missing_values=df.isnull().sum()
missing_values

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

##  Step 5: Feature Engineering
Feature transformations performed:
- Converted `Weekend` from Boolean to Numerical (0/1)
- Converted `Revenue` (Target Variable) to Binary (0/1)
- Created a new feature `Returning_Visitor` from VisitorType
- Dropped the original VisitorType column

These transformations improve model compatibility and performance.

In [5]:
#feature engieering  on weekend and Revenue column 
df["Weekend"]=df["Weekend"].replace((True,False),(1,0))
df["Revenue"]=df["Revenue"].replace((True,False),(1,0))
condition=df["VisitorType"]=="Returning_Visitor"



##  Step 6: Encoding Categorical Features
The `Month` column is encoded using Ordinal Encoding to convert categorical values into numerical format for machine learning models.

In [6]:
#adding the Returning_Visitor columns
df['Returning_Visitor']=np.where(condition,1,0)
df = df.drop(columns="VisitorType")
print(df.columns)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend',
       'Revenue', 'Returning_Visitor'],
      dtype='object')


In [7]:
#applying the one hot-encoding on moth column
encoding=OrdinalEncoder()
df["Month"]=encoding.fit_transform(df[["Month"]])
print(df["Month"])

0        2.0
1        2.0
2        2.0
3        2.0
4        2.0
        ... 
12325    1.0
12326    7.0
12327    7.0
12328    7.0
12329    7.0
Name: Month, Length: 12330, dtype: float64


##  Step 7: Correlation Analysis
We analyze the correlation of all features with the target variable `Revenue` 
to understand which features influence purchase behavior the most.

In [8]:
result = df.corr()['Revenue']						
result1 = result.sort_values(ascending=False)
print(result1)

Revenue                    1.000000
PageValues                 0.492569
ProductRelated             0.158538
ProductRelated_Duration    0.152373
Administrative             0.138917
Informational              0.095200
Administrative_Duration    0.093587
Month                      0.080150
Informational_Duration     0.070345
Weekend                    0.029295
Browser                    0.023984
TrafficType               -0.005113
Region                    -0.011595
OperatingSystems          -0.014668
SpecialDay                -0.082305
Returning_Visitor         -0.103843
BounceRates               -0.150673
ExitRates                 -0.207071
Name: Revenue, dtype: float64


##  Step 8: Data Preparation
- Features (X): All independent variables
- Target (Y): Revenue (Purchase = 1, No Purchase = 0)

This step separates input features and target variable for model training.

In [9]:
#data preparation
X=df.drop("Revenue",axis="columns")
Y=df["Revenue"]


##  Step 9: Train-Test Split
The dataset is split into training and testing sets:
- 70% Training Data
- 30% Testing Data

This helps evaluate the model on unseen data.

In [10]:
#preaparing the train and test dataset
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.3,random_state=0)




## Step 10: Building the ML Pipeline
A machine learning pipeline is created using:
- ColumnTransformer for preprocessing
- Numerical Pipeline (Imputation + Scaling)
- Categorical Pipeline (One-Hot Encoding)
- Model Integration

This ensures automated and reproducible preprocessing + training.

In [ ]:
def model_pipeline(x, model):

    # numerical & categorical columns
    n_c = x.select_dtypes(exclude=["object"]).columns.to_list()
    c_c = x.select_dtypes(include=["object"]).columns.to_list()

    # numerical pipeline
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant")),
        ("scaler", MinMaxScaler())
    ])

    # categorical pipeline
    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # column transformer
    preprocessing = ColumnTransformer([
        ("numeric", numerical_pipeline, n_c),
        ("categorical", categorical_pipeline, c_c)
    ])

    # final pipeline (SMOTE + model)
    final_step = [
        ("preprocessor", preprocessing),

        ("model", model)
    ]

    return IMBPipeline(steps=final_step)

##  Step 11: Model Selection using Cross Validation
Multiple machine learning models are evaluated using:
- 10-Fold Cross Validation
- ROC-AUC Score (Primary Metric)

Models Used:
- Random Forest Classifier
- Decision Tree Classifier
- K-Nearest Neighbors
- Bernoulli Naive Bayes
- Support Vector Classifier (SVC)

The best model is selected based on highest ROC-AUC score.

In [12]:
#model selection
def select_model(x,y,pipeline=None):
    classifiers={}
    c_d1={"RandomForestClassifier":RandomForestClassifier()}
    classifiers.update(c_d1)
    c_d2={"DecisionTreeClassifier":DecisionTreeClassifier()}
    classifiers.update(c_d2)
    c_d3={"KNeighborsClassifier": KNeighborsClassifier()}
    classifiers.update(c_d3)
    c_d4={"BernoulliNB":BernoulliNB()}
    classifiers.update(c_d4)
    c_df5={"SVC":SVC(probability=True)}
    classifiers.update(c_df5)
    c_df6={"RidgeClassifier":RidgeClassifier()}
    classifiers.update(c_df6)
    cols=["model","run_time","roc_auc"]
    df_models=pd.DataFrame(columns=cols)
    for key in classifiers:
        start_time=time.time()
        print()
        print("step 12:model_pipline run successfuly on",key)
        pipline=model_pipeline(x,classifiers[key])
        cv=cross_val_score(pipline,x,y,cv=10,scoring="roc_auc")
        row={
            "model":key,
            "run_time": format(round((time.time()-start_time)/60,2)),
            "roc_auc":cv.mean(),
        }
        df_models=pd.concat([df_models,pd.DataFrame([row])],ignore_index=True)
        df_models=df_models.sort_values(by="roc_auc",ascending=False)
        return df_models

##  Step 12: Training the Best Model
The selected best-performing model is trained using the full training dataset through the pipeline.

In [13]:
#Access Model select_model function
models=select_model(x_train,y_train)



step 12:model_pipline run successfuly on RandomForestClassifier


In [14]:
#toal model score 
print(models)

                    model run_time   roc_auc
0  RandomForestClassifier     0.12  0.925656


In [15]:
selected_model=SVC()
bundled_pipeline=model_pipeline(x_train,selected_model)
bundled_pipeline.fit(x_train,y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatti

In [16]:
#accessing best models 
y_pred=bundled_pipeline.predict(x_test)

##  Step 13: Model Evaluation
The trained model is evaluated using:
- ROC AUC Score
- Accuracy Score
- F1 Score
- Classification Report

These metrics help measure model performance and prediction quality.

In [ ]:
#Roc and Aoc score
sc = roc_auc_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print()
print('ROC/AUC:', sc)
print('Accuracy:', accuracy)
print('F1 score:', f1)






ROC/AUC: 0.6307802312980761
Accuracy: 0.8648283319816167
F1 score: 0.408983451536643
<class 'function'>


In [18]:
#Classification report #



classif_report = classification_report(y_test, y_pred)

print(classif_report)

              precision    recall  f1-score   support

           0       0.87      0.98      0.92      3077
           1       0.77      0.28      0.41       622

    accuracy                           0.86      3699
   macro avg       0.82      0.63      0.67      3699
weighted avg       0.85      0.86      0.84      3699



##  Conclusion
- Built an end-to-end ML pipeline for purchase prediction
- Performed feature engineering and preprocessing
- Used cross-validation for robust model selection
- Evaluated model using ROC-AUC, Accuracy, and F1 Score

### Key Outcome:
The pipeline successfully predicts online shopper purchase intention with optimized performance.

### Project Use Case:
This model can help e-commerce companies:
- Predict customer purchase behavior
- Improve marketing targeting
- Increase conversion rates